In [1]:
import os
import sys
import openai


from langchain.chains import ConversationalRetrievalChain, RetrievalQA
from langchain.chat_models import ChatOpenAI
from langchain.document_loaders import DirectoryLoader, TextLoader
from langchain.embeddings import OpenAIEmbeddings
from langchain.indexes import VectorstoreIndexCreator
from langchain.indexes.vectorstore import VectorStoreIndexWrapper
from langchain.llms import OpenAI
from langchain.vectorstores import Chroma
from transformers import pipeline, set_seed, GPT2LMHeadModel, GPT2Tokenizer

import constants

In [2]:
os.environ['OPENAI_API_KEY'] = ""

In [3]:
# Enable to save to disk & reuse the model (for repeated queries on the same data)
PERSIST = False

In [4]:
import openai

# Set your OpenAI API key here
openai.api_key = os.environ['OPENAI_API_KEY']


def load_sdg_summaries(file_path):
    with open(file_path, 'r') as file:
        sdg_summaries = file.read()
    return sdg_summaries

# Assuming 'sdg_summaries.txt' is in the same directory as your script
sdg_summaries = load_sdg_summaries('/Users/drishtant/Documents/Masters/sdg_summary.txt')

# # Use 'sdg_summaries' in your application as needed
# import json
# sdg_summaries = json.loads(sdg_summaries)


# Example 1

In [7]:
import openai

# Set your OpenAI API key here
openai.api_key = os.environ['OPENAI_API_KEY']

# Example SDG summaries as context (simplified for demonstration)
sdg_summaries = {
    'SDG_7': 'Ensure access to affordable, reliable, sustainable, and modern energy for all. Key objectives include increasing renewable energy use and improving energy efficiency.',
    'SDG_8' : 'Promote sustained, inclusive, and sustainable economic growth. Achieve full and productive employment and decent work for all.Ensure equal pay for equal work and safe working environments.',
    'SDG_13': 'Take urgent action to combat climate change and its impacts. Key objectives include strengthening resilience and adaptive capacity to climate-related hazards.',
    # Add other SDGs as needed...
}

def suggest_sdgs_for_project(project_description, sdg_keys):
    """
    Suggest relevant SDGs for a project description using GPT-3.5 Turbo.
    
    Args:
    - project_description (str): The description of the carbon credit project.
    - sdg_keys (list): List of SDG keys that are considered relevant for contextual information.
    
    Returns:
    - str: The model's suggestion for relevant SDGs.
    """
    # Concatenate SDG summaries for included SDGs
    context = " ".join([sdg_summaries[key] for key in sdg_keys])
    
    # Craft the prompt with both context and the project description
    prompt = f"{context} Given this context, analyze the following project description and suggest which UN SDGs it contributes to, provide list of un sdg as well as suggested sdg (not necessary if not required) for this project with proper expnanation : {project_description}"
    
    # Updated API call structure for openai library version 1.0.0 or later
    response = openai.ChatCompletion.create(
        model="gpt-3.5-turbo",
        messages=[
            {"role": "system", "content": "You are a GPT for rule based monitoring system that verify and monitor UN SDGs claimed by carbon credit project with proper analytical approach by using un sdg guidelines. You work as a human bot who can identify if UN SDGs claimed by project owner is correct or not, if not correct then provide suggestion based on project description. also Provide your confidence score for each un sdg claims."},
            {"role": "user", "content": prompt}
        ]
    )
    
    return response['choices'][0]['message']['content'].strip()
#     return response




In [8]:
# Example usage
project_description = "The Rimba Raya Biodiversity Reserve Project, an initiative by InfiniteEARTH, aims to reduce Indonesia‚Äö√Ñ√¥s emissions by preserving some 64,000 hectares of tropical peat swamp forest. This area, rich in biodiversity including the endangered Bornean orangutan, was slated by the Provincial government to be converted into four palm oil estates. Located on the southern coast of Borneo in the province of Central Kalimantan, the project is also designed to protect the integrity of the adjacent world-renowned Tanjung Puting National Park, by creating a physical buffer zone on the full extent of the ~90km eastern border of the park."
sdg_keys = ['SDG_7', 'SDG_13','SDG_8']  # Assuming relevance to SDGs 7,8 and 13 for context

suggestions = suggest_sdgs_for_project(project_description, sdg_keys)
print(suggestions)

Based on the project description provided, the Rimba Raya Biodiversity Reserve Project contributes to the following UN Sustainable Development Goals (SDGs):

1. **SDG 13: Climate Action**
   - The project directly contributes to this goal by reducing Indonesia's emissions through the preservation of 64,000 hectares of tropical peat swamp forest. By preventing deforestation and conversion of the area into palm oil estates, the project helps combat climate change and its impacts.

2. **SDG 15: Life on Land**
   - The project contributes to the conservation of biodiversity, particularly the endangered Bornean orangutan, by preserving the tropical peat swamp forest. This aligns with the objective of protecting and restoring terrestrial ecosystems and halting biodiversity loss.

3. **SDG 8: Decent Work and Economic Growth**
   - The project indirectly promotes sustainable economic growth by creating employment opportunities related to the conservation and management of the biodiversity rese

Based on the project description provided, the Rimba Raya Biodiversity Reserve Project contributes to the following UN Sustainable Development Goals (SDGs):

1. **SDG 13: Climate Action**
   - The project directly contributes to this goal by reducing Indonesia's emissions through the preservation of 64,000 hectares of tropical peat swamp forest. By preventing deforestation and conversion of the area into palm oil estates, the project helps combat climate change and its impacts.

2. **SDG 15: Life on Land**
   - The project contributes to the conservation of biodiversity, particularly the endangered Bornean orangutan, by preserving the tropical peat swamp forest. This aligns with the objective of protecting and restoring terrestrial ecosystems and halting biodiversity loss.

3. **SDG 8: Decent Work and Economic Growth**
   - The project indirectly promotes sustainable economic growth by creating employment opportunities related to the conservation and management of the biodiversity reserve. It also aligns with the objective of ensuring decent work for all and providing safe working environments.

**Suggested SDGs** (if required based on the project description):

1. **SDG 2: Zero Hunger**
   - There could be a potential indirect contribution to this goal if the project involves sustainable agriculture practices or alternative livelihood options for local communities to reduce dependency on activities contributing to hunger or food insecurity.

**Confidence Score**:
1. SDG 13: Climate Action - High confidence (9/10)
2. SDG 15: Life on Land - High confidence (9/10)
3. SDG 8: Decent Work and Economic Growth - Moderate confidence (7/10)
4. SDG 2: Zero Hunger (Suggested) - Low confidence (5/10)

In [9]:
sdg_summaries = {
    'SDG_1': {
        'description': 'No Poverty',
        'key_objectives': [
            'Eradicate extreme poverty.',
            'Implement social protection systems.',
            'Ensure equal rights to economic resources.'
        ],
        'scope': 'Focus on poverty reduction initiatives within carbon credit projects. Assess the extent to which projects contribute to lifting communities out of poverty.',
        'verification_criteria': {
            'Income Levels': [
                'Verify whether the project has led to increased income levels for communities involved.',
                'Evaluate changes in household income over time.'
            ],
            'Access to Basic Services': [
                'Determine if the project has improved access to basic services like healthcare, education, and clean water.',
                'Assess changes in infrastructure and service availability.'
            ],
            'Social Protection Systems': [
                'Evaluate the implementation of social protection programs within the project.',
                'Assess the effectiveness of safety nets for vulnerable populations.'
            ]
        }
    },
    'SDG_2': {
        'description': 'Zero Hunger',
        'key_objectives': [
            'End hunger and improve food security.',
            'Promote sustainable agriculture.',
            'Ensure access to nutritious food for all.'
        ],
        'scope': 'Assess efforts within carbon credit projects to promote sustainable agriculture and improve food security.',
        'verification_criteria': {
            'Food Security': [
                'Verify if the project has increased food availability and access for local communities.',
                'Assess changes in food production and distribution systems.'
            ],
            'Nutrition Improvement': [
                'Evaluate if the project has improved nutritional outcomes for participants.',
                'Assess changes in dietary diversity and nutritional status.'
            ],
            'Sustainable Agriculture': [
                'Determine if the project promotes sustainable agricultural practices.',
                'Assess changes in land use, crop diversity, and soil health.'
            ]
        }
    },
    'SDG_3': {
        'description': 'Good Health and Well-being',
        'key_objectives': [
            'Ensure access to healthcare services.',
            'Reduce maternal and child mortality.',
            'Combat communicable diseases.'
        ],
        'scope': 'Focus on initiatives within carbon credit projects that contribute to improved health outcomes.',
        'verification_criteria': {
            'Healthcare Access': [
                'Verify if the project has improved access to healthcare services for communities.',
                'Assess changes in healthcare infrastructure and service provision.'
            ],
            'Disease Prevention': [
                'Evaluate if the project has contributed to the prevention and control of diseases.',
                'Assess changes in disease incidence rates and public health initiatives.'
            ],
            'Mental Health Promotion': [
                'Determine if the project includes measures to promote mental health and well-being.',
                'Assess changes in community resilience and mental health awareness.'
            ]
        }
    },
    'SDG_4': {
        'description': 'Quality Education',
        'key_objectives': [
            'Ensure inclusive and equitable education for all.',
            'Improve literacy and numeracy skills.',
            'Enhance teacher training and educational facilities.'
        ],
        'scope': 'Assess initiatives within carbon credit projects aimed at promoting access to quality education and lifelong learning opportunities.',
        'verification_criteria': {
            'Educational Access': [
                'Verify if the project has increased access to formal and informal education for all, especially marginalized groups.',
                'Assess changes in enrollment rates and educational infrastructure.'
            ],
            'Quality of Education': [
                'Determine if the project has improved the quality of education provided.',
                'Assess changes in learning outcomes, teacher training, and curriculum development.'
            ],
            'Lifelong Learning': [
                'Evaluate if the project supports lifelong learning opportunities for community members.',
                'Assess access to vocational training, adult education, and skill-building programs.'
            ]
        }
    },
    'SDG_5': {
        'description': 'Gender Equality',
        'key_objectives': [
            'End discrimination and violence against women.',
            'Ensure equal opportunities for leadership and decision-making.',
            'Provide access to reproductive health services.'
        ],
        'scope': 'Focus on initiatives within carbon credit projects aimed at promoting gender equality and empowering women and girls.',
        'verification_criteria': {
            'Equal Participation': [
                'Verify if the project promotes equal participation and leadership opportunities for women and girls.',
                'Assess representation in decision-making processes and community initiatives.'
            ],
            'Access to Resources': [
                'Determine if the project improves access to resources and economic opportunities for women.',
                'Assess changes in land ownership, financial inclusion, and access to credit.'
            ],
            'Elimination of Gender-Based Violence': [
                'Evaluate if the project addresses gender-based violence and discrimination.',
                'Assess changes in attitudes towards gender equality and prevalence of violence.'
            ]
        }
    },
    'SDG_6': {
        'description': 'Clean Water and Sanitation',
        'key_objectives': [
            'Ensure access to clean water for all.',
            'Improve water quality and wastewater treatment.',
            'Enhance water use efficiency and conservation.'
        ],
        'scope': 'Assess efforts within carbon credit projects to ensure access to clean water and sanitation facilities.',
        'verification_criteria': {
            'Access to Safe Water': [
                'Verify if the project has improved access to safe and affordable drinking water for communities.',
                'Assess changes in water source infrastructure and water quality.'
            ],
            'Sanitation Facilities': [
                'Determine if the project has improved access to adequate sanitation facilities.',
                'Assess changes in toilet facilities, hygiene practices, and wastewater management.'
            ],
            'Water Resource Management': [
                'Evaluate if the project promotes sustainable water resource management practices.',
                'Assess changes in water conservation measures and ecosystem health.'
            ]
        }
    },
    'SDG_7': {
        'description': 'Affordable and Clean Energy',
        'key_objectives': [
            'Ensure access to affordable, reliable, sustainable, and modern energy for all.',
            'Increase the share of renewable energy in the global energy mix.',
            'Improve energy efficiency and infrastructure.'
        ],
        'scope': 'Assess initiatives within carbon credit projects that promote access to affordable, reliable, sustainable, and modern energy for all.',
        'verification_criteria': {
            'Energy Access': [
                'Verify if the project has increased access to electricity and clean cooking solutions for communities.',
                'Assess changes in energy infrastructure and grid connectivity.'
            ],
            'Renewable Energy Adoption': [
                'Determine if the project promotes the adoption of renewable energy sources.',
                'Assess the percentage increase in renewable energy use within project areas.'
            ],
            'Energy Efficiency': [
                'Evaluate if the project includes measures to improve energy efficiency.',
                'Assess changes in energy consumption patterns and efficiency measures implemented.'
            ]
        }
    },
    'SDG_8': {
        'description': 'Decent Work and Economic Growth',
        'key_objectives': [
            'Promote sustained, inclusive, and sustainable economic growth.',
            'Achieve full and productive employment and decent work for all.',
            'Ensure equal pay for equal work and safe working environments.'
        ],
        'scope': 'Focus on initiatives within carbon credit projects that promote inclusive and sustainable economic growth, full and productive employment, and decent work for all.',
        'verification_criteria': {
            'Job Creation': [
                'Verify if the project has created employment opportunities, particularly for marginalized groups.',
                'Assess changes in employment rates and types of jobs created.'
            ],
            'Income Generation': [
                'Determine if the project has contributed to increased incomes and economic opportunities for participants.',
                'Assess changes in household incomes and economic activities.'
            ],
            'Sustainable Business Practices': [
                'Evaluate if the project promotes sustainable business practices and entrepreneurship.',
                'Assess changes in business models, value chains, and market access for local enterprises.'
            ]
        }
    },
    'SDG_9': {
        'description': 'Industry, Innovation, and Infrastructure',
        'key_objectives': [
            'Develop resilient infrastructure to support economic growth and human well-being.',
            'Promote inclusive and sustainable industrialization.',
            'Foster innovation and technological advancements.'
        ],
        'scope': 'Assess initiatives within carbon credit projects that promote inclusive and sustainable industrialization, innovation, and infrastructure development.',
        'verification_criteria': {
            'Infrastructure Development': [
                'Verify if the project has improved infrastructure development, including transportation, water, and sanitation facilities.',
                'Assess changes in infrastructure quality and accessibility.'
            ],
            'Innovation Promotion': [
                'Determine if the project encourages innovation and technological advancement.',
                'Assess the adoption of innovative solutions and technologies within project activities.'
            ],
            'Sustainable Industrialization': [
                'Evaluate if the project promotes sustainable industrial practices and resource efficiency.',
                'Assess changes in industrial processes, waste management, and resource utilization.'
            ]
        }
    },
    'SDG_10': {
        'description': 'Reduced Inequalities',
        'key_objectives': [
            'Reduce income inequality within and among countries.',
            'Empower and promote social, economic, and political inclusion of all.',
            'Ensure equal opportunities for all, regardless of age, sex, disability, race, ethnicity, origin, religion, or economic or other status.'
        ],
        'scope': 'Focus on initiatives within carbon credit projects aimed at reducing inequalities within and among countries.',
        'verification_criteria': {
            'Income Equality': [
                'Verify if the project contributes to reducing income disparities within communities.',
                'Assess changes in income distribution and economic inclusion.'
            ],
            'Social Inclusion': [
                'Determine if the project promotes the social, economic, and political inclusion of all, regardless of age, sex, disability, race, ethnicity, origin, religion, or economic or other status.',
                'Assess changes in community engagement and representation.'
            ],
            'Access to Opportunities': [
                'Evaluate if the project ensures equal access to opportunities and resources.',
                'Assess changes in access to education, healthcare, and employment opportunities.'
            ]
        }
    },
    'SDG_11': {
        'description': 'Sustainable Cities and Communities',
        'key_objectives': [
            'Make cities inclusive, safe, resilient, and sustainable.',
            'Provide access to safe and affordable housing and transportation.',
            'Protect cultural and natural heritage.'
        ],
        'scope': 'Focus on urban sustainability initiatives within carbon credit projects. Assess efforts to create inclusive, safe, resilient, and sustainable cities and communities.',
        'verification_criteria': {
            'Urban Infrastructure': [
                'Verify if the project contributes to the development of sustainable urban infrastructure.',
                'Assess improvements in transportation, housing, and public spaces.'
            ],
            'Community Engagement': [
                'Determine if the project fosters community participation in decision-making processes.',
                'Assess the level of community involvement in planning and implementation.'
            ],
            'Resilience Building': [
                'Evaluate if the project enhances urban resilience to natural and man-made disasters.',
                'Assess measures taken to mitigate risks and adapt to climate change impacts.'
            ]
        }
    },
    'SDG_12': {
        'description': 'Responsible Consumption and Production',
        'key_objectives': [
            'Ensure sustainable consumption and production patterns.',
            'Reduce waste generation and promote recycling.',
            'Encourage sustainable resource management practices.'
        ],
        'scope': 'Assess initiatives within carbon credit projects aimed at promoting responsible consumption and production patterns.',
        'verification_criteria': {
            'Resource Efficiency': [
                'Verify if the project promotes resource-efficient practices along the production and consumption chain.',
                'Assess reductions in resource use and waste generation.'
            ],
            'Circular Economy Principles': [
                'Determine if the project adopts principles of a circular economy, including recycling and reuse.',
                'Assess changes in product design and waste management practices.'
            ],
            'Sustainable Supply Chains': [
                'Evaluate if the project encourages the adoption of sustainable sourcing and production methods.',
                'Assess changes in supply chain transparency and ethical practices.'
            ]
        }
    },
    'SDG_13': {
        'description': 'Climate Action',
        'key_objectives': [
            'Strengthen resilience and adaptive capacity to climate-related hazards.',
            'Integrate climate change measures into national policies, strategies, and planning.',
            'Promote education, awareness-raising, and capacity-building on climate change mitigation and adaptation.'
        ],
        'scope': 'Focus on carbon credit projects\' contributions to mitigating climate change and building resilience to its impacts.',
        'verification_criteria': {
            'Emission Reductions': [
                'Verify if the project leads to measurable reductions in greenhouse gas emissions.',
                'Assess changes in carbon sequestration and emission avoidance.'
            ],
            'Adaptation Measures': [
                'Determine if the project includes measures to adapt to climate change impacts.',
                'Assess resilience-building initiatives and adaptation strategies.'
            ],
            'Climate Education and Awareness': [
                'Evaluate if the project promotes climate education and awareness among stakeholders.',
                'Assess changes in knowledge, attitudes, and behaviors related to climate change.'
            ]
        }
    },
    'SDG_14': {
        'description': 'Life Below Water',
        'key_objectives': [
            'Conserve and sustainably use marine resources.',
            'Protect and restore marine ecosystems and biodiversity.',
            'Reduce marine pollution and ocean acidification.'
        ],
        'scope': 'Focus on initiatives within carbon credit projects aimed at preserving marine ecosystems and biodiversity.',
        'verification_criteria': {
            'Marine Conservation': [
                'Verify if the project contributes to the conservation of marine habitats and species.',
                'Assess changes in marine biodiversity and ecosystem health.'
            ],
            'Sustainable Fisheries': [
                'Determine if the project promotes sustainable fishing practices.',
                'Assess changes in fish stocks and fishing methods to reduce environmental impact.'
            ],
            'Pollution Reduction': [
                'Evaluate if the project addresses marine pollution and its sources.',
                'Assess changes in pollution levels and waste management practices.'
            ]
        }
    },
    'SDG_15': {
        'description': 'Life on Land',
        'key_objectives': [
            'Protect, restore, and promote sustainable use of terrestrial ecosystems.',
            'Combat desertification, halt land degradation, and biodiversity loss.',
            'Ensure the conservation of forests, mountains, and other terrestrial ecosystems.'
        ],
        'scope': 'Assess initiatives within carbon credit projects focused on land conservation, reforestation, and biodiversity preservation.',
        'verification_criteria': {
            'Forest Conservation': [
                'Verify if the project contributes to the conservation of forests and terrestrial ecosystems.',
                'Assess changes in forest cover, biodiversity, and ecosystem services.'
            ],
            'Reforestation and Afforestation': [
                'Determine if the project includes efforts to restore degraded lands through reforestation or afforestation.',
                'Assess changes in vegetation cover and habitat restoration.'
            ],
            'Biodiversity Protection': [
                'Evaluate if the project safeguards biodiversity and endangered species.',
                'Assess changes in species richness and habitat quality.'
            ]
        }
    },
    'SDG_16': {
        'description': 'Peace, Justice, and Strong Institutions',
        'key_objectives': [
            'Promote peaceful and inclusive societies for sustainable development.',
            'Ensure access to justice for all and build effective, accountable, and inclusive institutions at all levels.',
            'Reduce violence, corruption, and illicit financial and arms flows.'
        ],
        'scope': 'Focus on initiatives within carbon credit projects that promote peace, justice, and inclusive societies.',
        'verification_criteria': {
            'Conflict Resolution': [
                'Verify if the project contributes to conflict resolution and peacebuilding efforts.',
                'Assess changes in community cohesion and conflict resolution mechanisms.'
            ],
            'Access to Justice': [
                'Determine if the project improves access to justice and legal rights for marginalized groups.',
                'Assess changes in legal empowerment and dispute resolution mechanisms.'
            ],
            'Good Governance': [
                'Evaluate if the project strengthens governance structures and promotes transparency.',
                'Assess changes in accountability mechanisms and community participation in decision-making.'
            ]
        }
    },
    'SDG_17': {
        'description': 'Partnerships for the Goals',
        'key_objectives': [
            'Strengthen global partnerships for sustainable development.',
            'Mobilize resources to support implementation of the SDGs.',
            'Enhance policy coherence and global collaboration.'
        ],
        'scope': 'Assess partnerships and collaborations within carbon credit projects to support sustainable development.',
        'verification_criteria': {
            'Stakeholder Engagement': [
                'Verify if the project engages a diverse range of stakeholders, including government, civil society, and private sector actors.',
                'Assess the effectiveness of collaboration and partnership-building efforts.'
            ],
            'Resource Mobilization': [
                'Determine if the project mobilizes resources effectively to support sustainable development initiatives.',
                'Assess changes in funding sources and investment in sustainable projects.'
            ],
            'Knowledge Sharing': [
                'Evaluate if the project facilitates knowledge sharing and capacity building among stakeholders.',
                'Assess changes in information exchange and learning opportunities.'
            ]
        }
    }
}

# Example 2

In [10]:
def suggest_sdgs_for_project(project_description, sdg_keys):
    """
    Suggest relevant SDGs for a project description using GPT-3.5 Turbo.
    
    Args:
    - project_description (str): The description of the carbon credit project.
    - sdg_keys (list): List of SDG keys claimed by the project owner.
    
    Returns:
    - str: The model's suggestion for relevant SDGs.
    """
    # Concatenate SDG summaries for all SDGs
    all_sdg_context = " ".join([f"{key}: {sdg_summaries[key]['description']}. Key objectives: {'; '.join(sdg_summaries[key]['key_objectives'])}." for key in sdg_summaries.keys()])
    
    # Concatenate SDG summaries for claimed SDGs
    claimed_sdg_context = " ".join([f"{key}: {sdg_summaries[key]['description']}. Key objectives: {'; '.join(sdg_summaries[key]['key_objectives'])}." for key in sdg_keys])
    
    # Craft the prompt with both contexts and the project description
    prompt = f"Full SDG context: {all_sdg_context} Claimed SDG context: {claimed_sdg_context} Given this context, analyze the following project description and suggest which UN SDGs it contributes to. Provide a list of UN SDGs as well as suggested SDGs (not necessary if not required) for this project with proper explanation: {project_description}. Additionally, identify any discrepancies between the SDGs claimed by the project and the ones it should be associated with based on the project description. This is the claimed sdgs.{sdg_keys}"

    try:
        response = openai.ChatCompletion.create(
            model="gpt-4",
            temperature = 0.3,
            messages=[
                {"role": "system", "content": "You are a GPT for a rule-based monitoring system that verifies and monitors UN SDGs claimed by carbon credit projects with proper analytical approach using UN SDG guidelines. You work as a human bot who can identify if UN SDGs claimed by project owners are correct or not; if not correct, then provide suggestions based on the project description.also Provide your confidence score for each un sdg claims."},
                {"role": "user", "content": prompt}
            ]
        )
        return response['choices'][0]['message']['content'].strip()
    except Exception as e:
        return f"An error occurred: {str(e)}"

# Example usage
project_description = "The Rimba Raya Biodiversity Reserve Project, an initiative by InfiniteEARTH, aims to reduce Indonesia's emissions by preserving some 64,000 hectares of tropical peat swamp forest. This area, rich in biodiversity including the endangered Bornean orangutan, was slated by the Provincial government to be converted into four palm oil estates. Located on the southern coast of Borneo in the province of Central Kalimantan, the project is also designed to protect the integrity of the adjacent world-renowned Tanjung Puting National Park, by creating a physical buffer zone on the full extent of the ~90km eastern border of the park."
sdg_keys = ['SDG_1', 'SDG_2', 'SDG_3', 'SDG_4', 'SDG_5', 'SDG_6', 'SDG_7', 'SDG_8', 'SDG_9', 'SDG_10', 'SDG_11', 'SDG_12', 'SDG_13', 'SDG_14', 'SDG_15', 'SDG_16', 'SDG_17']  # Assuming relevance to all SDGs for context

suggestions = suggest_sdgs_for_project(project_description, sdg_keys)

print(suggestions)

Based on the project description, the Rimba Raya Biodiversity Reserve Project primarily contributes to the following UN SDGs:

1. SDG_13: Climate Action - The project aims to reduce Indonesia's emissions by preserving a large area of tropical peat swamp forest, which aligns with the goal's objective to strengthen resilience and adaptive capacity to climate-related hazards. Confidence Score: 90%

2. SDG_15: Life on Land - The project is designed to protect biodiversity, including the endangered Bornean orangutan, and prevent the conversion of the forest into palm oil estates. This aligns with the goal's objectives to protect, restore, and promote sustainable use of terrestrial ecosystems. Confidence Score: 95%

3. SDG_17: Partnerships for the Goals - The project is an initiative by InfiniteEARTH, which may involve partnerships with other organizations or stakeholders for its implementation. This aligns with the goal's objectives to strengthen global partnerships for sustainable developm

In [12]:
# Example usage
project_description = "The Rimba Raya Biodiversity Reserve Project, an initiative by InfiniteEARTH, aims to reduce Indonesia's emissions by preserving some 64,000 hectares of tropical peat swamp forest. This area, rich in biodiversity including the endangered Bornean orangutan, was slated by the Provincial government to be converted into four palm oil estates. Located on the southern coast of Borneo in the province of Central Kalimantan, the project is also designed to protect the integrity of the adjacent world-renowned Tanjung Puting National Park, by creating a physical buffer zone on the full extent of the ~90km eastern border of the park."
sdg_keys = ['SDG_1', 'SDG_2', 'SDG_3', 'SDG_4', 'SDG_5', 'SDG_6', 'SDG_7', 'SDG_8', 'SDG_9', 'SDG_10', 'SDG_11', 'SDG_12', 'SDG_13', 'SDG_14', 'SDG_15', 'SDG_16', 'SDG_17']  # Assuming relevance to all SDGs for context

suggestions = suggest_sdgs_for_project(project_description, sdg_keys)

print(suggestions)

Based on the project description, the Rimba Raya Biodiversity Reserve Project primarily contributes to the following UN SDGs:

1. SDG_13: Climate Action - The project aims to reduce Indonesia's emissions by preserving a significant area of tropical peat swamp forest, which is a key action in mitigating climate change. Confidence Score: 95%

2. SDG_15: Life on Land - The project directly contributes to this goal by protecting and promoting the sustainable use of terrestrial ecosystems. It also contributes to the conservation of forests and biodiversity, including the endangered Bornean orangutan. Confidence Score: 95%

3. SDG_17: Partnerships for the Goals - The project by InfiniteEARTH could be seen as a form of partnership for achieving the goal of sustainable development. However, without more specific information about collaborations or partnerships, the confidence score is lower. Confidence Score: 70%

The project does not provide sufficient evidence to support contributions to the

# Example 3 - One shot Prompt

In [13]:
import openai
import os

# Set your OpenAI API key here
openai.api_key = os.environ['OPENAI_API_KEY']

def suggest_sdgs_for_project(project_description, sdg_keys, sdg_summaries, model="gpt-4", temperature=0.3):
    """
    Suggest relevant SDGs for a project description using GPT-3.5 Turbo.
    
    Args:
    - project_description (str): The description of the carbon credit project.
    - sdg_keys (list): List of SDG keys claimed by the project owner.
    - sdg_summaries (dict): Dictionary containing the SDG summaries.
    - model (str): The model to use for the API call. Default is "gpt-3.5-turbo".
    - temperature (float): The temperature setting for the model to control randomness.
    
    Returns:
    - str: The model's suggestion for relevant SDGs.
    """
    # Concatenate SDG summaries for all SDGs
    all_sdg_context = " ".join([f"{key}: {sdg_summaries[key]['description']}. Key objectives: {'; '.join(sdg_summaries[key]['key_objectives'])}." for key in sdg_summaries.keys()])
    
    # Concatenate SDG summaries for claimed SDGs
    claimed_sdg_context = " ".join([f"{key}: {sdg_summaries[key]['description']}. Key objectives: {'; '.join(sdg_summaries[key]['key_objectives'])}." for key in sdg_keys])
    
    # Craft the one-shot prompt with an example and the project description
    example_project_description = "The Rimba Raya Biodiversity Reserve Project aims to reduce Indonesia's emissions by preserving some 64,000 hectares of tropical peat swamp forest."
    example_claimed_sdgs = ['SDG_13', 'SDG_15', 'SDG_17']
    example_suggestion = """
    The Rimba Raya Biodiversity Reserve Project contributes to the following UN Sustainable Development Goals (SDGs):

    1. **SDG 13: Climate Action**
       - The project helps in reducing emissions by preserving tropical peat swamp forest, which serves as a carbon sink and helps mitigate climate change.

    2. **SDG 15: Life on Land**
       - By preserving 64,000 hectares of tropical peat swamp forest, the project contributes to protecting terrestrial ecosystems, biodiversity, and endangered species.

    3. **SDG 17: Partnerships for the Goals**
       - The project highlights the importance of collaboration between organizations and governmental bodies to achieve sustainable development goals, emphasizing the need for global partnerships for conservation efforts.

    Based on the project description, the claimed SDGs are appropriate, but additional contributions to SDG 14: Life Below Water could be considered if there are efforts to protect adjacent marine ecosystems.
    """

    prompt = f"""
    Full SDG context: {all_sdg_context}
    Claimed SDG context: {claimed_sdg_context}

    Example:
    Project Description: {example_project_description}
    Claimed SDGs: {example_claimed_sdgs}
    Suggestion: {example_suggestion}

    Now, analyze the following project description and suggest which UN SDGs it contributes to. Provide a list of UN SDGs as well as suggested SDGs for this project with proper explanation. Additionally, identify any discrepancies between the SDGs claimed by the project and the ones it should be associated with based on the project description.

    Project Description: {project_description}
    Claimed SDGs: {sdg_keys}
    """

    try:
        response = openai.ChatCompletion.create(
            model=model,
            temperature=temperature,
            messages=[
                {"role": "system", "content": "You are a GPT for a rule-based monitoring system that verifies and monitors UN SDGs claimed by carbon credit projects with proper analytical approach using UN SDG guidelines. You work as a human bot who can identify if UN SDGs claimed by project owners are correct or not; if not correct, then provide suggestions based on the project description.also Provide your confidence score for each un sdg claims."},
                {"role": "user", "content": prompt}
            ]
        )
        return response['choices'][0]['message']['content'].strip()
    except Exception as e:
        return f"An error occurred: {str(e)}"

In [14]:
# Example usage
project_description = "The Rimba Raya Biodiversity Reserve Project, an initiative by InfiniteEARTH, aims to reduce Indonesia's emissions by preserving some 64,000 hectares of tropical peat swamp forest. This area, rich in biodiversity including the endangered Bornean orangutan, was slated by the Provincial government to be converted into four palm oil estates. Located on the southern coast of Borneo in the province of Central Kalimantan, the project is also designed to protect the integrity of the adjacent world-renowned Tanjung Puting National Park, by creating a physical buffer zone on the full extent of the ~90km eastern border of the park."
sdg_keys = ['SDG_1', 'SDG_2', 'SDG_3', 'SDG_4', 'SDG_5', 'SDG_6', 'SDG_7', 'SDG_8', 'SDG_9', 'SDG_10', 'SDG_11', 'SDG_12', 'SDG_13', 'SDG_14', 'SDG_15', 'SDG_16', 'SDG_17']  # Assuming relevance to all SDGs for context
suggestions = suggest_sdgs_for_project(project_description, sdg_keys, sdg_summaries)

print(suggestions)

Based on the project description, the Rimba Raya Biodiversity Reserve Project contributes to the following UN Sustainable Development Goals (SDGs):

1. **SDG 13: Climate Action**
   - The project helps in reducing emissions by preserving tropical peat swamp forest, which serves as a carbon sink and helps mitigate climate change.

2. **SDG 15: Life on Land**
   - By preserving 64,000 hectares of tropical peat swamp forest, the project contributes to protecting terrestrial ecosystems, biodiversity, and endangered species like the Bornean orangutan.

3. **SDG 17: Partnerships for the Goals**
   - The project highlights the importance of collaboration between organizations and governmental bodies to achieve sustainable development goals, emphasizing the need for global partnerships for conservation efforts.

The project also indirectly contributes to:

4. **SDG 14: Life Below Water**
   - By protecting the integrity of the adjacent Tanjung Puting National Park and creating a physical buffe

Based on the project description, the Rimba Raya Biodiversity Reserve Project contributes to the following UN Sustainable Development Goals (SDGs):

1. **SDG 13: Climate Action**
   - The project helps in reducing emissions by preserving tropical peat swamp forest, which serves as a carbon sink and helps mitigate climate change.

2. **SDG 15: Life on Land**
   - By preserving 64,000 hectares of tropical peat swamp forest, the project contributes to protecting terrestrial ecosystems, biodiversity, and endangered species like the Bornean orangutan.

3. **SDG 17: Partnerships for the Goals**
   - The project highlights the importance of collaboration between organizations and governmental bodies to achieve sustainable development goals, emphasizing the need for global partnerships for conservation efforts.

The project also indirectly contributes to:

4. **SDG 14: Life Below Water**
   - By protecting the integrity of the adjacent Tanjung Puting National Park and creating a physical buffer zone, the project may also contribute to the conservation of marine ecosystems in the area, although this is not explicitly stated in the project description.

However, the project does not provide specific information to support the claimed contributions to SDGs 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, and 16. Without additional information on how the project directly contributes to these goals, it is not possible to confirm their relevance.

Confidence Score:
- SDG 13: 90% (High confidence based on the project's focus on emission reduction and climate change mitigation)
- SDG 15: 90% (High confidence based on the project's focus on biodiversity conservation and ecosystem protection)
- SDG 17: 75% (Moderate confidence based on the project's implied need for partnerships, but without explicit details on collaborative efforts)
- SDG 14: 50% (Low confidence as the project's contribution to this goal is inferred and not explicitly stated)
- SDGs 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 16: 10% (Very low confidence due to lack of specific information in the project description)

# Example : Few Shot 

In [15]:

def suggest_sdgs_for_project(project_description, sdg_keys, sdg_summaries, model="gpt-4", temperature=0.3):
    """
    Suggest relevant SDGs for a project description using GPT-3.5 Turbo.
    
    Args:
    - project_description (str): The description of the carbon credit project.
    - sdg_keys (list): List of SDG keys claimed by the project owner.
    - sdg_summaries (dict): Dictionary containing the SDG summaries.
    - model (str): The model to use for the API call. Default is "gpt-3.5-turbo".
    - temperature (float): The temperature setting for the model to control randomness.
    
    Returns:
    - str: The model's suggestion for relevant SDGs.
    """
    # Concatenate SDG summaries for all SDGs
    all_sdg_context = " ".join([f"{key}: {sdg_summaries[key]['description']}. Key objectives: {'; '.join(sdg_summaries[key]['key_objectives'])}." for key in sdg_summaries.keys()])
    
    # Concatenate SDG summaries for claimed SDGs
    claimed_sdg_context = " ".join([f"{key}: {sdg_summaries[key]['description']}. Key objectives: {'; '.join(sdg_summaries[key]['key_objectives'])}." for key in sdg_keys])
    
    # Few-shot examples
    examples = """
    Example 1:
    Project Description: A wind power project in Maharashtra aims to generate renewable energy and reduce carbon emissions.
    Claimed SDGs: ['SDG_7', 'SDG_13']
    Suggestion: 
    The Grid Connected Wind Power Project in Maharashtra contributes to the following UN Sustainable Development Goals (SDGs):
    1. **SDG 7: Affordable and Clean Energy**
       - The project generates renewable energy, reducing reliance on fossil fuels.
    2. **SDG 13: Climate Action**
       - The project helps mitigate climate change by reducing carbon emissions.
    Based on the project description, the claimed SDGs are appropriate.

    Example 2:
    Project Description: A 300 MW wind energy project contributes to renewable energy generation and carbon emission reduction.
    Claimed SDGs: ['SDG_7', 'SDG_13', 'SDG_15']
    Suggestion:
    The 300 MW Wind Energy Project contributes to the following UN Sustainable Development Goals (SDGs):
    1. **SDG 7: Affordable and Clean Energy**
       - The project generates significant amounts of renewable energy.
    2. **SDG 13: Climate Action**
       - The project reduces carbon emissions, aiding in climate mitigation.
    3. **SDG 15: Life on Land**
       - The project helps preserve local ecosystems by reducing pollution.
    Based on the project description, the claimed SDGs are appropriate.

    Example 3:
    Project Description: The project promotes the use of improved cookstoves in Nigeria to reduce indoor air pollution and carbon emissions.
    Claimed SDGs: ['SDG_3', 'SDG_7', 'SDG_13']
    Suggestion:
    The Promoting Improved Cooking Practices in Nigeria project contributes to the following UN Sustainable Development Goals (SDGs):
    1. **SDG 3: Good Health and Well-being**
       - The project reduces indoor air pollution, improving respiratory health.
    2. **SDG 7: Affordable and Clean Energy**
       - The project promotes the use of efficient cookstoves, reducing energy consumption.
    3. **SDG 13: Climate Action**
       - The project helps mitigate climate change by reducing carbon emissions.
    Based on the project description, the claimed SDGs are appropriate.
    """

    prompt = f"""
    Full SDG context: {all_sdg_context}
    Claimed SDG context: {claimed_sdg_context}
    {examples}

    Now, analyze the following project description and suggest which UN SDGs it contributes to. Provide a list of UN SDGs as well as suggested SDGs for this project with proper explanation. Additionally, identify any discrepancies between the SDGs claimed by the project and the ones it should be associated with based on the project description.

    Project Description: {project_description}
    Claimed SDGs: {sdg_keys}
    """

    try:
        response = openai.ChatCompletion.create(
            model=model,
            temperature=temperature,
            messages=[
                {"role": "system", "content": "You are a GPT for a rule-based monitoring system that verifies and monitors UN SDGs claimed by carbon credit projects with proper analytical approach using UN SDG guidelines. You work as a human bot who can identify if UN SDGs claimed by project owners are correct or not; if not correct, then provide suggestions based on the project description.also Provide your confidence score for each un sdg claims."},
                {"role": "user", "content": prompt}
            ]
        )
        return response['choices'][0]['message']['content'].strip()
    except Exception as e:
        return f"An error occurred: {str(e)}"


In [16]:
# Example usage
project_description = "The Rimba Raya Biodiversity Reserve Project, an initiative by InfiniteEARTH, aims to reduce Indonesia's emissions by preserving some 64,000 hectares of tropical peat swamp forest. This area, rich in biodiversity including the endangered Bornean orangutan, was slated by the Provincial government to be converted into four palm oil estates. Located on the southern coast of Borneo in the province of Central Kalimantan, the project is also designed to protect the integrity of the adjacent world-renowned Tanjung Puting National Park, by creating a physical buffer zone on the full extent of the ~90km eastern border of the park."
sdg_keys = ['SDG_1', 'SDG_2', 'SDG_3', 'SDG_4', 'SDG_5', 'SDG_6', 'SDG_7', 'SDG_8', 'SDG_9', 'SDG_10', 'SDG_11', 'SDG_12', 'SDG_13', 'SDG_14', 'SDG_15', 'SDG_16', 'SDG_17']  # Assuming relevance to all SDGs for context
suggestions = suggest_sdgs_for_project(project_description, sdg_keys, sdg_summaries)

print(suggestions)

Based on the project description, the Rimba Raya Biodiversity Reserve Project contributes to the following UN Sustainable Development Goals (SDGs):

1. **SDG 13: Climate Action**
   - The project helps mitigate climate change by reducing Indonesia's emissions through the preservation of a significant area of tropical peat swamp forest. 

2. **SDG 15: Life on Land**
   - The project protects, restores, and promotes sustainable use of terrestrial ecosystems. It preserves an area rich in biodiversity, including the endangered Bornean orangutan. It also protects the integrity of the adjacent Tanjung Puting National Park by creating a physical buffer zone.

The project has claimed all 17 SDGs, which seems to be an overestimation based on the provided project description. The project primarily contributes to SDG 13 and SDG 15. There is no clear evidence in the project description to suggest direct contributions to other SDGs such as No Poverty (SDG 1), Zero Hunger (SDG 2), Good Health and We

Based on the project description, the Rimba Raya Biodiversity Reserve Project contributes to the following UN Sustainable Development Goals (SDGs):

1. **SDG 13: Climate Action**
   - The project helps mitigate climate change by reducing Indonesia's emissions through the preservation of a significant area of tropical peat swamp forest. 

2. **SDG 15: Life on Land**
   - The project protects, restores, and promotes sustainable use of terrestrial ecosystems. It preserves an area rich in biodiversity, including the endangered Bornean orangutan. It also protects the integrity of the adjacent Tanjung Puting National Park by creating a physical buffer zone.

The project has claimed all 17 SDGs, which seems to be an overestimation based on the provided project description. The project primarily contributes to SDG 13 and SDG 15. There is no clear evidence in the project description to suggest direct contributions to other SDGs such as No Poverty (SDG 1), Zero Hunger (SDG 2), Good Health and Well-being (SDG 3), Quality Education (SDG 4), Gender Equality (SDG 5), Clean Water and Sanitation (SDG 6), Affordable and Clean Energy (SDG 7), Decent Work and Economic Growth (SDG 8), Industry, Innovation, and Infrastructure (SDG 9), Reduced Inequalities (SDG 10), Sustainable Cities and Communities (SDG 11), Responsible Consumption and Production (SDG 12), Life Below Water (SDG 14), Peace, Justice, and Strong Institutions (SDG 16), and Partnerships for the Goals (SDG 17).

Therefore, the project should revise its claimed SDGs to accurately reflect its contributions. The project may contribute to other SDGs indirectly or in ways not mentioned in the provided description, but based on the available information, the primary contributions are to SDG 13 and SDG 15.

Confidence score for each UN SDG claim:

- SDG 13: Climate Action - 90%
- SDG 15: Life on Land - 90%
- All other SDGs - 10%

# Example 5 : Chain of Thoughts


In [17]:

def suggest_sdgs_for_project(project_description, sdg_keys, sdg_summaries, model="gpt-4", temperature=0.3):
    """
    Suggest relevant SDGs for a project description using GPT-3.5 Turbo.
    
    Args:
    - project_description (str): The description of the carbon credit project.
    - sdg_keys (list): List of SDG keys claimed by the project owner.
    - sdg_summaries (dict): Dictionary containing the SDG summaries.
    - model (str): The model to use for the API call. Default is "gpt-3.5-turbo".
    - temperature (float): The temperature setting for the model to control randomness.
    
    Returns:
    - str: The model's suggestion for relevant SDGs.
    """
    # Concatenate SDG summaries for all SDGs
    all_sdg_context = " ".join([f"{key}: {sdg_summaries[key]['description']}. Key objectives: {'; '.join(sdg_summaries[key]['key_objectives'])}." for key in sdg_summaries.keys()])
    
    # Concatenate SDG summaries for claimed SDGs
    claimed_sdg_context = " ".join([f"{key}: {sdg_summaries[key]['description']}. Key objectives: {'; '.join(sdg_summaries[key]['key_objectives'])}." for key in sdg_keys])
    
    # Few-shot examples
    # Few-shot examples with chain of thought
    examples = """
    Example 1:
    Project Description: A wind power project in Maharashtra aims to generate renewable energy and reduce carbon emissions.
    Claimed SDGs: ['SDG_7', 'SDG_13']
    Chain of Thought:
    - This project generates renewable energy, which aligns with SDG 7: Affordable and Clean Energy.
    - By reducing reliance on fossil fuels and lowering carbon emissions, it also contributes to SDG 13: Climate Action.
    Suggestion: 
    The Grid Connected Wind Power Project in Maharashtra contributes to the following UN Sustainable Development Goals (SDGs):
    1. **SDG 7: Affordable and Clean Energy**
       - The project generates renewable energy, reducing reliance on fossil fuels.
    2. **SDG 13: Climate Action**
       - The project helps mitigate climate change by reducing carbon emissions.
    Based on the project description, the claimed SDGs are appropriate.

    Example 2:
    Project Description: A 300 MW wind energy project contributes to renewable energy generation and carbon emission reduction.
    Claimed SDGs: ['SDG_7', 'SDG_13', 'SDG_15']
    Chain of Thought:
    - This project generates significant amounts of renewable energy, which aligns with SDG 7: Affordable and Clean Energy.
    - By reducing carbon emissions, it supports SDG 13: Climate Action.
    - The project might also have an impact on local ecosystems, aligning with SDG 15: Life on Land.
    Suggestion:
    The 300 MW Wind Energy Project contributes to the following UN Sustainable Development Goals (SDGs):
    1. **SDG 7: Affordable and Clean Energy**
       - The project generates significant amounts of renewable energy.
    2. **SDG 13: Climate Action**
       - The project reduces carbon emissions, aiding in climate mitigation.
    3. **SDG 15: Life on Land**
       - The project helps preserve local ecosystems by reducing pollution.
    Based on the project description, the claimed SDGs are appropriate.

    Example 3:
    Project Description: The project promotes the use of improved cookstoves in Nigeria to reduce indoor air pollution and carbon emissions.
    Claimed SDGs: ['SDG_3', 'SDG_7', 'SDG_13']
    Chain of Thought:
    - This project reduces indoor air pollution, improving respiratory health, which aligns with SDG 3: Good Health and Well-being.
    - Promoting the use of efficient cookstoves aligns with SDG 7: Affordable and Clean Energy.
    - By reducing carbon emissions, the project supports SDG 13: Climate Action.
    Suggestion:
    The Promoting Improved Cooking Practices in Nigeria project contributes to the following UN Sustainable Development Goals (SDGs):
    1. **SDG 3: Good Health and Well-being**
       - The project reduces indoor air pollution, improving respiratory health.
    2. **SDG 7: Affordable and Clean Energy**
       - The project promotes the use of efficient cookstoves, reducing energy consumption.
    3. **SDG 13: Climate Action**
       - The project helps mitigate climate change by reducing carbon emissions.
    Based on the project description, the claimed SDGs are appropriate.
    """

    prompt = f"""
    Full SDG context: {all_sdg_context}
    Claimed SDG context: {claimed_sdg_context}
    {examples}

    Now, analyze the following project description and suggest which UN SDGs it contributes to. Provide a list of UN SDGs as well as suggested SDGs for this project with proper explanation. Additionally, identify any discrepancies between the SDGs claimed by the project and the ones it should be associated with based on the project description. Use a chain of thought approach to explain your reasoning.

    Project Description: {project_description}
    Claimed SDGs: {sdg_keys}
    """

    try:
        response = openai.ChatCompletion.create(
            model=model,
            temperature=temperature,
            messages=[
                {"role": "system", "content": "You are a GPT for a rule-based monitoring system that verifies and monitors UN SDGs claimed by carbon credit projects with proper analytical approach using UN SDG guidelines. You work as a human bot who can identify if UN SDGs claimed by project owners are correct or not; if not correct, then provide suggestions based on the project description.also Provide your confidence score for each un sdg claims."},
                {"role": "user", "content": prompt}
            ]
        )
        return response['choices'][0]['message']['content'].strip()
    except Exception as e:
        return f"An error occurred: {str(e)}"

In [18]:
# Example usage
project_description = "The Rimba Raya Biodiversity Reserve Project, an initiative by InfiniteEARTH, aims to reduce Indonesia's emissions by preserving some 64,000 hectares of tropical peat swamp forest. This area, rich in biodiversity including the endangered Bornean orangutan, was slated by the Provincial government to be converted into four palm oil estates. Located on the southern coast of Borneo in the province of Central Kalimantan, the project is also designed to protect the integrity of the adjacent world-renowned Tanjung Puting National Park, by creating a physical buffer zone on the full extent of the ~90km eastern border of the park."
sdg_keys = ['SDG_1', 'SDG_2', 'SDG_3', 'SDG_4', 'SDG_5', 'SDG_6', 'SDG_7', 'SDG_8', 'SDG_9', 'SDG_10', 'SDG_11', 'SDG_12', 'SDG_13', 'SDG_14', 'SDG_15', 'SDG_16', 'SDG_17']  # Assuming relevance to all SDGs for context
suggestions = suggest_sdgs_for_project(project_description, sdg_keys, sdg_summaries)

print(suggestions)

Chain of Thought:
- The project aims to reduce carbon emissions by preserving a large area of tropical peat swamp forest, which aligns with SDG 13: Climate Action.
- By preserving the forest, the project is also protecting biodiversity, including endangered species, which aligns with SDG 15: Life on Land.
- The project prevents the conversion of the forest into palm oil estates, which can be seen as a measure against unsustainable agricultural practices, aligning with SDG 12: Responsible Consumption and Production.
- The project creates a physical buffer zone for a national park, which can be seen as a measure to protect natural heritage, aligning with SDG 11: Sustainable Cities and Communities.

Suggestion:
The Rimba Raya Biodiversity Reserve Project contributes to the following UN Sustainable Development Goals (SDGs):
1. **SDG 13: Climate Action**
   - The project reduces carbon emissions by preserving a large area of tropical peat swamp forest.
2. **SDG 15: Life on Land**
   - The p

Chain of Thought:
- The project aims to reduce carbon emissions by preserving a large area of tropical peat swamp forest, which aligns with SDG 13: Climate Action.
- By preserving the forest, the project is also protecting biodiversity, including endangered species, which aligns with SDG 15: Life on Land.
- The project prevents the conversion of the forest into palm oil estates, which can be seen as a measure against unsustainable agricultural practices, aligning with SDG 12: Responsible Consumption and Production.
- The project creates a physical buffer zone for a national park, which can be seen as a measure to protect natural heritage, aligning with SDG 11: Sustainable Cities and Communities.

Suggestion:
The Rimba Raya Biodiversity Reserve Project contributes to the following UN Sustainable Development Goals (SDGs):
1. **SDG 13: Climate Action**
   - The project reduces carbon emissions by preserving a large area of tropical peat swamp forest.
2. **SDG 15: Life on Land**
   - The project protects biodiversity, including endangered species, by preserving their natural habitat.
3. **SDG 12: Responsible Consumption and Production**
   - The project prevents the conversion of the forest into palm oil estates, promoting sustainable agricultural practices.
4. **SDG 11: Sustainable Cities and Communities**
   - The project protects natural heritage by creating a physical buffer zone for a national park.

Based on the project description, the claimed SDGs seem to be too broad. While the project may indirectly contribute to some of the other SDGs, the direct impact is most closely aligned with SDGs 13, 15, 12, and 11. Therefore, it is suggested that the project should primarily claim these four SDGs.

Confidence Score:
- SDG 13: Climate Action: 90%
- SDG 15: Life on Land: 90%
- SDG 12: Responsible Consumption and Production: 80%
- SDG 11: Sustainable Cities and Communities: 80%

# Example 5 : Contrastive Chain of Thoughts


In [19]:
def suggest_sdgs_for_project(project_description, sdg_keys, sdg_summaries, model="gpt-4", temperature=0.3):
    """
    Suggest relevant SDGs for a project description using GPT-3.5 Turbo.
    
    Args:
    - project_description (str): The description of the carbon credit project.
    - sdg_keys (list): List of SDG keys claimed by the project owner.
    - sdg_summaries (dict): Dictionary containing the SDG summaries.
    - model (str): The model to use for the API call. Default is "gpt-3.5-turbo".
    - temperature (float): The temperature setting for the model to control randomness.
    
    Returns:
    - str: The model's suggestion for relevant SDGs.
    """
    # Concatenate SDG summaries for all SDGs
    all_sdg_context = " ".join([f"{key}: {sdg_summaries[key]['description']}. Key objectives: {'; '.join(sdg_summaries[key]['key_objectives'])}." for key in sdg_summaries.keys()])
    
    # Concatenate SDG summaries for claimed SDGs
    claimed_sdg_context = " ".join([f"{key}: {sdg_summaries[key]['description']}. Key objectives: {'; '.join(sdg_summaries[key]['key_objectives'])}." for key in sdg_keys])
    
    # Few-shot examples with contrastive chain of thought
    examples = """
    Example 1:
    Project Description: A wind power project in Maharashtra aims to generate renewable energy and reduce carbon emissions.
    Claimed SDGs: ['SDG_7', 'SDG_13']
    Chain of Thought:
    - Positive: This project generates renewable energy, which aligns with SDG 7: Affordable and Clean Energy.
    - Positive: By reducing reliance on fossil fuels and lowering carbon emissions, it also contributes to SDG 13: Climate Action.
    - Contrast: The project does not seem to impact other SDGs directly, such as SDG 1 (No Poverty) or SDG 3 (Good Health and Well-being).
    Suggestion: 
    The Grid Connected Wind Power Project in Maharashtra contributes to the following UN Sustainable Development Goals (SDGs):
    1. **SDG 7: Affordable and Clean Energy**
       - The project generates renewable energy, reducing reliance on fossil fuels.
    2. **SDG 13: Climate Action**
       - The project helps mitigate climate change by reducing carbon emissions.
    Based on the project description, the claimed SDGs are appropriate.

    Example 2:
    Project Description: A 300 MW wind energy project contributes to renewable energy generation and carbon emission reduction.
    Claimed SDGs: ['SDG_7', 'SDG_13', 'SDG_15']
    Chain of Thought:
    - Positive: This project generates significant amounts of renewable energy, which aligns with SDG 7: Affordable and Clean Energy.
    - Positive: By reducing carbon emissions, it supports SDG 13: Climate Action.
    - Positive: The project might also have an impact on local ecosystems, aligning with SDG 15: Life on Land.
    - Contrast: The project does not seem to address SDGs related to poverty, health, or education directly.
    Suggestion:
    The 300 MW Wind Energy Project contributes to the following UN Sustainable Development Goals (SDGs):
    1. **SDG 7: Affordable and Clean Energy**
       - The project generates significant amounts of renewable energy.
    2. **SDG 13: Climate Action**
       - The project reduces carbon emissions, aiding in climate mitigation.
    3. **SDG 15: Life on Land**
       - The project helps preserve local ecosystems by reducing pollution.
    Based on the project description, the claimed SDGs are appropriate.

    Example 3:
    Project Description: The project promotes the use of improved cookstoves in Nigeria to reduce indoor air pollution and carbon emissions.
    Claimed SDGs: ['SDG_3', 'SDG_7', 'SDG_13']
    Chain of Thought:
    - Positive: This project reduces indoor air pollution, improving respiratory health, which aligns with SDG 3: Good Health and Well-being.
    - Positive: Promoting the use of efficient cookstoves aligns with SDG 7: Affordable and Clean Energy.
    - Positive: By reducing carbon emissions, the project supports SDG 13: Climate Action.
    - Contrast: The project does not seem to directly impact SDGs related to education, infrastructure, or inequality.
    Suggestion:
    The Promoting Improved Cooking Practices in Nigeria project contributes to the following UN Sustainable Development Goals (SDGs):
    1. **SDG 3: Good Health and Well-being**
       - The project reduces indoor air pollution, improving respiratory health.
    2. **SDG 7: Affordable and Clean Energy**
       - The project promotes the use of efficient cookstoves, reducing energy consumption.
    3. **SDG 13: Climate Action**
       - The project helps mitigate climate change by reducing carbon emissions.
    Based on the project description, the claimed SDGs are appropriate.
    """

    prompt = f"""
    Full SDG context: {all_sdg_context}
    Claimed SDG context: {claimed_sdg_context}
    {examples}

    Now, analyze the following project description and suggest which UN SDGs it contributes to. Provide a list of UN SDGs as well as suggested SDGs for this project with proper explanation. Additionally, identify any discrepancies between the SDGs claimed by the project and the ones it should be associated with based on the project description. Use a contrastive chain of thought approach to explain your reasoning.

    Project Description: {project_description}
    Claimed SDGs: {sdg_keys}
    """

    try:
        response = openai.ChatCompletion.create(
            model=model,
            temperature=temperature,
            messages=[
                {"role": "system", "content": "You are a GPT for a rule-based monitoring system that verifies and monitors UN SDGs claimed by carbon credit projects with proper analytical approach using UN SDG guidelines. You work as a human bot who can identify if UN SDGs claimed by project owners are correct or not; if not correct, then provide suggestions based on the project description.also Provide your confidence score for each un sdg claims."},
                {"role": "user", "content": prompt}
            ]
        )
        return response['choices'][0]['message']['content'].strip()
    except Exception as e:
        return f"An error occurred: {str(e)}"

In [20]:
project_description = "The Rimba Raya Biodiversity Reserve Project, an initiative by InfiniteEARTH, aims to reduce Indonesia's emissions by preserving some 64,000 hectares of tropical peat swamp forest. This area, rich in biodiversity including the endangered Bornean orangutan, was slated by the Provincial government to be converted into four palm oil estates. Located on the southern coast of Borneo in the province of Central Kalimantan, the project is also designed to protect the integrity of the adjacent world-renowned Tanjung Puting National Park, by creating a physical buffer zone on the full extent of the ~90km eastern border of the park."
sdg_keys = ['SDG_1', 'SDG_2', 'SDG_3', 'SDG_4', 'SDG_5', 'SDG_6', 'SDG_7', 'SDG_8', 'SDG_9', 'SDG_10', 'SDG_11', 'SDG_12', 'SDG_13', 'SDG_14', 'SDG_15', 'SDG_16', 'SDG_17']  # Assuming relevance to all SDGs for context
suggestions = suggest_sdgs_for_project(project_description, sdg_keys, sdg_summaries)

print(suggestions)

Chain of Thought:
- Positive: The project preserves a significant area of tropical peat swamp forest, which aligns with SDG 15: Life on Land.
- Positive: By preserving the forest, the project reduces carbon emissions, contributing to SDG 13: Climate Action.
- Positive: The project also protects biodiversity, including endangered species, which aligns with SDG 15: Life on Land.
- Positive: The project prevents the conversion of the forest into palm oil estates, which can be seen as promoting sustainable use of terrestrial ecosystems and aligns with SDG 12: Responsible Consumption and Production.
- Contrast: The project does not seem to directly impact SDGs related to poverty (SDG 1), hunger (SDG 2), health (SDG 3), education (SDG 4), gender equality (SDG 5), clean water and sanitation (SDG 6), affordable and clean energy (SDG 7), decent work and economic growth (SDG 8), industry, innovation, and infrastructure (SDG 9), reduced inequalities (SDG 10), sustainable cities and communities (S

Chain of Thought:
- Positive: The project preserves a significant area of tropical peat swamp forest, which aligns with SDG 15: Life on Land.
- Positive: By preserving the forest, the project reduces carbon emissions, contributing to SDG 13: Climate Action.
- Positive: The project also protects biodiversity, including endangered species, which aligns with SDG 15: Life on Land.
- Positive: The project prevents the conversion of the forest into palm oil estates, which can be seen as promoting sustainable use of terrestrial ecosystems and aligns with SDG 12: Responsible Consumption and Production.
- Contrast: The project does not seem to directly impact SDGs related to poverty (SDG 1), hunger (SDG 2), health (SDG 3), education (SDG 4), gender equality (SDG 5), clean water and sanitation (SDG 6), affordable and clean energy (SDG 7), decent work and economic growth (SDG 8), industry, innovation, and infrastructure (SDG 9), reduced inequalities (SDG 10), sustainable cities and communities (SDG 11), life below water (SDG 14), peace, justice, and strong institutions (SDG 16), and partnerships for the goals (SDG 17).

Suggestion:
The Rimba Raya Biodiversity Reserve Project contributes to the following UN Sustainable Development Goals (SDGs):
1. **SDG 13: Climate Action**
   - The project helps mitigate climate change by reducing carbon emissions through forest preservation.
2. **SDG 15: Life on Land**
   - The project protects, restores, and promotes sustainable use of terrestrial ecosystems, especially by protecting the endangered Bornean orangutan.
3. **SDG 12: Responsible Consumption and Production**
   - The project prevents the conversion of the forest into palm oil estates, promoting sustainable resource management practices.

Based on the project description, the claimed SDGs seem to be too broad. The project primarily contributes to SDGs 13, 15, and 12. The other SDGs claimed by the project are not directly addressed in the project description provided.

Confidence Score:
- SDG 13: Climate Action: 90%
- SDG 15: Life on Land: 90%
- SDG 12: Responsible Consumption and Production: 80%
